# Data audit

Loads Belgian day-ahead electricity prices and checks schema, timestamps, missing values and units.


## Data contract

The file must contain:

| Column | Meaning | Expected type |
|---|---|---|
| `timestamp` | Start of the hourly delivery period | Time-zone-aware datetime |
| `price_eur_per_mwh` | Price for that hour | Numeric, in EUR/MWh |

The audit checks the following before any modelling:

- Required columns exist
- Timestamps parse correctly and retain time-zone information
- No missing prices or duplicate timestamps
- Timestamp spacing is hourly for this sample
- Price range and negative-price count are reported

In [1]:
from pathlib import Path

import pandas as pd

# Resolve the data path relative to the repository root.
data_path = Path("data/belgium_day_ahead_sample.csv")

# Parse the ISO-8601 timestamps while preserving their UTC offset.
prices = pd.read_csv(data_path, parse_dates=["timestamp"])

prices.head()


,timestamp,price_eur_per_mwh
0,2025-01-06 00:00:00+01:00,61.2
1,2025-01-06 01:00:00+01:00,57.8
2,2025-01-06 02:00:00+01:00,52.1
3,2025-01-06 03:00:00+01:00,48.7
4,2025-01-06 04:00:00+01:00,45.9


## Basic schema checks


In [2]:
required_columns = {"timestamp", "price_eur_per_mwh"}
missing_columns = required_columns - set(prices.columns)

assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"
assert pd.api.types.is_datetime64_any_dtype(prices["timestamp"]), "timestamp was not parsed as datetime"
assert pd.api.types.is_numeric_dtype(prices["price_eur_per_mwh"]), "price must be numeric"
assert prices["timestamp"].dt.tz is not None, "timestamp must include a time zone or UTC offset"

print("Schema checks passed.")
print(prices.dtypes)


Schema checks passed.
timestamp            datetime64[ns, UTC+01:00]
price_eur_per_mwh                      float64
dtype: object


## Timestamp and missing-data checks


In [3]:
# Sort first so consecutive timestamp differences are meaningful.
prices = prices.sort_values("timestamp").reset_index(drop=True)

missing_price_count = prices["price_eur_per_mwh"].isna().sum()
duplicate_timestamp_count = prices["timestamp"].duplicated().sum()
hourly_steps = prices["timestamp"].diff().dropna().eq(pd.Timedelta(hours=1))
non_hourly_step_count = (~hourly_steps).sum()

assert missing_price_count == 0, f"Found {missing_price_count} missing prices"
assert duplicate_timestamp_count == 0, f"Found {duplicate_timestamp_count} duplicate timestamps"
assert non_hourly_step_count == 0, f"Found {non_hourly_step_count} non-hourly timestamp gaps"

print(f"Rows: {len(prices)}")
print(f"Period: {prices['timestamp'].min()} to {prices['timestamp'].max()}")
print("Timestamp and missing-data checks passed.")


Rows: 24
Period: 2025-01-06 00:00:00+01:00 to 2025-01-06 23:00:00+01:00
Timestamp and missing-data checks passed.


## Price profile


In [4]:
price_summary = prices["price_eur_per_mwh"].agg(["count", "min", "max", "mean", "median", "std"])
negative_price_count = (prices["price_eur_per_mwh"] < 0).sum()
lowest_price_row = prices.loc[prices["price_eur_per_mwh"].idxmin()]
highest_price_row = prices.loc[prices["price_eur_per_mwh"].idxmax()]

print("Price summary (EUR/MWh):")
print(price_summary.round(2))
print(f"Negative-price hours: {negative_price_count}")
print(f"Lowest price: {lowest_price_row['price_eur_per_mwh']:.2f} EUR/MWh at {lowest_price_row['timestamp']}")
print(f"Highest price: {highest_price_row['price_eur_per_mwh']:.2f} EUR/MWh at {highest_price_row['timestamp']}")


Price summary (EUR/MWh):
count      24.00
min        45.90
max       128.70
mean       74.13
median     67.00
std        23.44
Name: price_eur_per_mwh, dtype: float64
Negative-price hours: 0
Lowest price: 45.90 EUR/MWh at 2025-01-06 04:00:00+01:00
Highest price: 128.70 EUR/MWh at 2025-01-06 18:00:00+01:00


## Audit record
